# 찾아줘 독립 식단 크롤러 v0.3.1 (라이브 점검판)

**식샤 코드를 실행하거나 식샤 API를 호출하지 않습니다.** 우리 코드로 원천 사이트를 수집하고, 향후 30일 날짜별 메뉴 + 식당/코너별 위치·운영시간 + DB 동기화를 실습합니다.

기본 설정은 **`MODE="demo"`**이며 독립적으로 제작한 합성 HTML만 사용합니다. 실제 원천 사이트는 외부 접속, 사이트 이용조건과 접근 허가를 확인하고 명시적으로 `"live"`로 바꾼 경우에만 요청합니다.

In [ ]:
from pathlib import Path
import sys, json, sqlite3
from datetime import date, datetime
from zoneinfo import ZoneInfo
import pandas as pd
HERE=Path.cwd().resolve()
ROOT=next((p for p in (HERE, HERE.parent, HERE.parent.parent) if (p / "src/chajwo_meals").is_dir()),None)
if ROOT is None:
    raise RuntimeError("노트북은 프로젝트 폴더의 notebooks/ 또는 루트에서 실행해 주세요.")
sys.path.insert(0,str(ROOT/'src'))
from chajwo_meals.pipeline import collect, collect_metadata, export, flatten_entries, daterange
from chajwo_meals.directory import make_directory, parse_guide, parse_campus_map
from chajwo_meals.hours import extract_service_hours
from chajwo_meals.database import sync_db
from chajwo_meals.fixed import load_fixed_catalog, fixed_item_rows
from chajwo_meals.parser import classify_line
print('PROJECT ROOT:', ROOT)

## 1. 실행 모드 설정

`DAYS=30`은 오늘부터 30개 날짜를 순회한다는 의미이며, 아직 게시되지 않은 미래 식단은 결측으로 남습니다. `MODE="live"`는 실제 SNUCO·SNUDORM·VET·공식 식당안내·캠퍼스맵에 접속합니다. **MODE만 바꾸면 KST 오늘 날짜를 자동 선택하고, 별도의 `outputs/notebook_live_v031/live_only.sqlite3`에 저장합니다.** 인터넷/DNS 또는 원천 서버 상태에 따라 실제 수집이 실패할 수 있으며, 완료 메시지만으로 성공을 판단하면 안 됩니다.

In [ ]:
MODE='demo'   # 필요할 때만 'live'로 수정
START_DATE=(date(2026,10,22) if MODE=='demo' else datetime.now(ZoneInfo('Asia/Seoul')).date()) # live: 실행일(KST) 자동 사용
DAYS=30
SOURCES=('snuco','snudorm','vet')
OUTPUT=ROOT/'outputs'/('notebook_demo_v031' if MODE=='demo' else 'notebook_live_v031')
OUTPUT.mkdir(parents=True,exist_ok=True)
DB_PATH=OUTPUT/('demo_only.sqlite3' if MODE=='demo' else 'live_only.sqlite3')
DB_URL='sqlite:///'+str(DB_PATH)
print('모드:', MODE, '/ 출력:', OUTPUT, '/ DB:', DB_PATH)
print(f'{START_DATE} ~ {daterange(START_DATE,DAYS)[-1]} / {DAYS}일')

## 2. 직접 수집 흐름

데모용 `FixtureFetcher`는 HTTP 요청 대신 `tests/fixtures`에 별도 제작한 테스트 HTML을 반환합니다. 날짜 확인 로직을 테스트하기 위해 각 날짜의 input 값을 변경합니다.

In [ ]:
class FixtureFetcher:
    def __init__(self,root):
        self.root=root
        self.calls=[]
    def get(self,url):
        self.calls.append(url)
        if 'snuco.snu.ac.kr/foodmenu' in url:
            html=(self.root/'snuco.html').read_text(encoding='utf8')
        elif 'snudorm.snu.ac.kr/foodmenu' in url:
            html=(self.root/'snudorm.html').read_text(encoding='utf8')
        elif 'vet.snu.ac.kr/cafe_menu' in url:
            return (self.root/'vet.html').read_text(encoding='utf8')
        elif '/식당안내/' in url:
            return (self.root/'guide_demo.html').read_text(encoding='utf8')
        elif '/api/building.action' in url:
            return (self.root/'campus_map_demo.json').read_text(encoding='utf8')
        else:
            raise ValueError('Unexpected URL: '+url)
        import re
        from urllib.parse import urlsplit,parse_qs
        day=parse_qs(urlsplit(url).query)['date'][0]
        return html.replace('2026-10-22',day)
if MODE=='demo':
    fetcher=FixtureFetcher(ROOT/'tests'/'fixtures')
else:
    from chajwo_meals.http import RateLimitedFetcher
    fetcher=RateLimitedFetcher(min_interval=2.0)
rows,errors=collect(start=START_DATE,days=DAYS,sources=SOURCES,fetcher=fetcher)
directory,meta_errors=collect_metadata(fetcher=fetcher)
errors.extend(meta_errors)
print('관측 줄:',len(rows),'수집 오류:',len(errors),'식당/코너 수:',len(directory))
if MODE=='demo': print('데모에서 대체한 URL 요청 횟수:',len(fetcher.calls))

## 3. 식당 등록부, 운영시간, 코너·장소 정보

식샤 V2 공개 SQL 초기 등록 31개 코너와 공식 안내 기준 신규 2개 식당(버거운버거, 프레퍼스)을 합친 33개 장소입니다. 데모에서는 공식 사이트 데이터가 아닌 **합성된 운영시간/좌표**를 사용합니다.

In [ ]:
directory_df=pd.DataFrame(directory)
display(directory_df[['restaurant_unit','building_number','floor','weekday_hours','latitude','hours_status']].head(12))
print('식샤 초기 seed 31개 / 공식 안내에서 신규 2개 / 현재 등록 33개:',len(directory))

## 4. 호구세트 8,300원과 가격 보정 검증

천 단위 쉼표의 마지막 그룹이 규격 밖이고 초과 부분이 0이면 끝자리 0을 제거합니다. 정상 가격에서는 0을 제거하지 않습니다.

In [ ]:
for s in ('호구세트 : 8,3000원','일반 식당 : 10,000원','표기 불명 : 8,3001원'):
    x=classify_line(s)
    print(s,'→',x['price_won'],'원 /',x['price_status'])
ho=[r for r in load_fixed_catalog() if r['menu_name']=='호구세트']
assert ho[0]['price_won']==8300
print('고정 목록 호구세트:',ho[0]['price_won'])

## 5. CSV/JSON 생성 및 SQLite DB 동기화

기본 데모에서는 `outputs/notebook_demo_v031/demo_only.sqlite3`에만 저장합니다. 운영 환경에서는 영속 PostgreSQL을 사용하고, 접근 실패 시 이전 정상 식단을 보존해야 합니다.

In [ ]:
report=export(rows,errors,OUTPUT,started=START_DATE,days=DAYS,sources=SOURCES,restaurant_directory=directory)
entries=flatten_entries([r for r in rows if not (r['source']=='snuco' and r['restaurant'] in ('버거운버거','공대간이식당'))])
entries.extend([{'entry_kind':'fixed_catalog',**r} for r in fixed_item_rows(load_fixed_catalog())])
db_report=sync_db(DB_URL,entries=entries,directory=directory,errors=errors,started=START_DATE.isoformat(),days=DAYS,
                  manifest=report,hour_rows=extract_service_hours(rows))
print(json.dumps({'report':report,'db':db_report},ensure_ascii=False,indent=2))
if MODE=='live':
    if errors:
        print('주의: 실제 수집 오류',len(errors),'건 - errors.json과 manifest.json을 확인하세요.')
    elif not rows:
        print('주의: 요청 오류는 없지만 신규 식단 행이 없습니다. 원천 게시 상태를 확인하세요.')
    else:
        print('실제 수집행:',len(rows),'DB 갱신 범위:',db_report['updated_source_date_scopes'])

In [ ]:
with sqlite3.connect(DB_PATH) as connection:
    status=connection.execute('SELECT COUNT(*) FROM restaurants').fetchone()[0]
    count=connection.execute('SELECT COUNT(*) FROM menu_entries').fetchone()[0]
    fixed=connection.execute('SELECT price_won FROM menu_entries WHERE menu_name=? AND data_origin=?',('호구세트','fixed_catalog')).fetchone()
print('DB 식당수:',status,'메뉴행:',count,'호구세트 확정값:',fixed)
display(pd.read_csv(OUTPUT/'siksha_seed_comparison.csv',encoding='utf-8-sig').tail(3))

## 6. DB·수집 검증과 실전 운영 가이드

- 실제 운영 자동 실행: `.github/workflows/full-month.yml` 05:20 KST 30일, `same-day-refresh.yml` 12:20·18:20·23:20 KST 당일+2일. GitHub Actions에는 **영속 PostgreSQL** Secret이 필요합니다.
- 출처에 게시되지 않은 미래 식단은 채워지지 않습니다.
- 시간이 확인되지 않은 식당은 `hours_status=unknown`, 건물 좌표가 없으면 NULL입니다.
- 크롤링 사이트 약관·재사용 허가를 확인한 후 운영에 배포합니다.
- 식샤 소스/API는 참조하지 않으며 프로젝트 독립 코드만 사용합니다.

In [ ]:
assert status==33
assert fixed and fixed[0]==8300
assert report['days']==30
assert len(daterange(START_DATE,30))==30
print('데모 검증 완료: 합성 원문입니다.' if MODE=='demo' else '라이브 실행 종료: manifest/errors와 DB 갱신 범위 확인이 필요합니다.')